# 🛡️ BƯỚC 1: HERETIC UNCENSORING PIPELINE (Chạy trên Colab GPU A100 / L4 / T4)
Notebook này thực hiện **loại bỏ 100% kiểm duyệt / từ chối trả lời (Censorship / Refusal)** cho các mô hình Coding bằng công cụ **Heretic**.

### 📌 Quy trình hoàn toàn tự động 100% (Pure Python Engine):
1. Tự động kiểm tra GPU A100/L4, dọn dẹp môi trường và cài đặt `heretic-llm`.
2. Kết nối Google Drive lưu trữ model.
3. Chạy trực tiếp thuật toán Heretic Uncensoring bằng Python Engine, tự động tìm kiếm vector triệt tiêu tối ưu qua Optuna, hợp nhất trọng số (Merge) và lưu thẳng vào Google Drive.
4. Kiểm tra toàn bộ file model đã hoàn tất trên Google Drive (`/content/drive/MyDrive/ai_coding_models_uncensored/`).
5. (Tùy chọn) Nhập Token Hugging Face để tự động đẩy model lên tài khoản của bạn.

In [ ]:
# @title 1. Tự động Tối ưu Môi trường & Cài đặt Heretic LLM
import torch
!nvidia-smi

# Gỡ bỏ torchaudio bị lệch CUDA phiên bản để tránh lỗi BloomPreTrainedModel
!pip uninstall -y torchaudio

# Cài đặt Heretic và các thư viện cần thiết
!pip install -q -U heretic-llm torch torchvision transformers accelerate bitsandbytes huggingface_hub optuna

In [ ]:
# @title 2. Kết nối Google Drive để lưu trữ Model
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=True)

# Tạo thư mục chứa các model uncensored trên Drive
SAVE_DIR = "/content/drive/MyDrive/ai_coding_models_uncensored"
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"📁 Thư mục lưu trữ model đã uncensor: {SAVE_DIR}")

In [ ]:
# @title 3. Chạy Heretic Uncensoring Tự Động (Pure Python Engine)
import os
import sys
import time
import math
import torch
import torch.nn.functional as F
import transformers
import optuna
from optuna.samplers import TPESampler
from dataclasses import asdict
from os.path import commonprefix

from heretic.config import Settings, QuantizationMethod
from heretic.model import Model, AbliterationParameters
from heretic.evaluator import Evaluator
from heretic.utils import load_prompts, get_trial_parameters, format_duration
from heretic.system import empty_cache

# Cấu hình tham số
MODEL_CHOICE = "Qwen/Qwen2.5-Coder-7B-Instruct" # @param ["Qwen/Qwen2.5-Coder-7B-Instruct", "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B", "deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct"]
USE_4BIT_QUANT = False # @param {type:"boolean"} # Khuyên dùng: False khi chạy trên A100 40GB/80GB để giữ độ chính xác cao nhất
N_TRIALS = 10 # @param {type:"integer"}

clean_name = MODEL_CHOICE.split("/")[-1]
OUTPUT_MODEL_NAME = f"{clean_name}-Heretic-Uncensored"
TARGET_PATH = os.path.join(SAVE_DIR, OUTPUT_MODEL_NAME)

print(f"🎯 Model mục tiêu: {MODEL_CHOICE}")
print(f"📁 Thư mục lưu trực tiếp trên Drive: {TARGET_PATH}")
print(f"⚡ Thiết bị GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / (1024**3):.1f} GB VRAM)")

# 1. Khởi tạo Settings Heretic
settings = Settings(
    model=MODEL_CHOICE,
    quantization=QuantizationMethod.BNB_4BIT if USE_4BIT_QUANT else QuantizationMethod.NONE,
    n_trials=N_TRIALS,
    n_startup_trials=min(3, max(1, N_TRIALS // 2))
)

torch.set_grad_enabled(False)
transformers.logging.set_verbosity_error()
optuna.logging.set_verbosity(optuna.logging.WARNING)

# 2. Tải Model & Bộ Dataset đối chiếu
print("\n📥 [1/4] Đang tải Model và Prompts kiểm duyệt...")
model = Model(settings)
good_prompts = load_prompts(settings, settings.good_prompts)
bad_prompts = load_prompts(settings, settings.bad_prompts)
print(f"✅ Đã tải: {len(good_prompts)} harmless prompts + {len(bad_prompts)} refusal prompts")

# Tối ưu hóa batch size
if settings.batch_size == 0:
    settings.batch_size = 16

# Kiểm tra prefix Chain-of-Thought
prefix_check_prompts = good_prompts[:100] + bad_prompts[:100]
responses = model.get_responses_batched(prefix_check_prompts)
settings.response_prefix = commonprefix(responses).rstrip(" ")
if settings.response_prefix:
    for cot_initializer, closed_cot_block in settings.chain_of_thought_skips:
        if settings.response_prefix.startswith(cot_initializer):
            settings.response_prefix = closed_cot_block
            break

# 3. Tính toán Residual Directions
print("\n🧮 [2/4] Đang tính toán không gian vector kiểm duyệt (Residual Directions)...")
good_means = model.get_residuals_mean(good_prompts)
bad_means = model.get_residuals_mean(bad_prompts)
residual_directions = F.normalize(bad_means - good_means, p=2, dim=1)

if settings.orthogonalize_direction:
    good_directions = F.normalize(good_means, p=2, dim=1)
    projection_vector = torch.sum(residual_directions * good_directions, dim=1)
    residual_directions = (
        residual_directions - projection_vector.unsqueeze(1) * good_directions
    )
    residual_directions = F.normalize(residual_directions, p=2, dim=1)
    del good_directions, projection_vector

del good_means, bad_means
empty_cache()

# 4. Optuna Pareto Optimization Search
evaluator = Evaluator(settings, model)
sampler = TPESampler(n_startup_trials=settings.n_startup_trials, multivariate=True, seed=42)
study = optuna.create_study(directions=["minimize"] * len(evaluator.get_objective_names()), sampler=sampler)

print(f"\n🚀 [3/4] Bắt đầu tìm kiếm siêu tham số tối ưu ({N_TRIALS} Trials)...")
last_layer_index = len(model.get_layers()) - 1
start_time = time.perf_counter()

for trial_num in range(1, N_TRIALS + 1):
    trial = study.ask()
    direction_scope = trial.suggest_categorical("direction_scope", ["global", "per layer"])
    direction_index = trial.suggest_float("direction_index", 0.4 * last_layer_index, 0.9 * last_layer_index)
    if direction_scope == "per layer":
        direction_index = None

    parameters = {}
    for component in model.get_abliterable_components():
        max_weight_lower_bound = -0.25 if component == "mlp.down_proj" else 0.8
        max_weight = max(0.0, trial.suggest_float(f"{component}.max_weight", max_weight_lower_bound, 1.5))
        max_weight_position = trial.suggest_float(f"{component}.max_weight_position", 0.6 * last_layer_index, 1.0 * last_layer_index)
        min_weight = trial.suggest_float(f"{component}.min_weight", 0.0, 1.0)
        min_weight_distance = trial.suggest_float(f"{component}.min_weight_distance", 1.0, max(0.6 * last_layer_index, 1.0))
        parameters[component] = AbliterationParameters(
            max_weight=max_weight,
            max_weight_position=max_weight_position,
            min_weight=min_weight * max_weight,
            min_weight_distance=min_weight_distance,
        )

    model.reset_model()
    model.abliterate(residual_directions, direction_index, parameters)
    scores = evaluator.get_scores()
    objective_values = evaluator.get_objective_values(scores)
    study.tell(trial, objective_values)

    trial.set_user_attr("direction_index", direction_index)
    trial.set_user_attr("parameters", {k: asdict(v) for k, v in parameters.items()})

    score_display = ", ".join([f"{name}: {score.rich_display}" for name, score in scores])
    elapsed = format_duration(time.perf_counter() - start_time)
    print(f"  • Trial {trial_num:02d}/{N_TRIALS:02d} [{elapsed}] -> {score_display}")

# 5. Chọn Trial tối ưu & Merge trực tiếp vào Google Drive
best_trial = min(study.best_trials, key=lambda t: t.values[0])
print(f"\n🏆 [4/4] Chọn Trial tối ưu nhất: Trial #{best_trial.number}")

best_dir_idx = best_trial.user_attrs["direction_index"]
best_params = {k: AbliterationParameters(**v) for k, v in best_trial.user_attrs["parameters"].items()}

model.reset_model()
model.abliterate(residual_directions, best_dir_idx, best_params)

print(f"\n💾 Đang merge trọng số và lưu model trực tiếp vào: {TARGET_PATH}...")
os.makedirs(TARGET_PATH, exist_ok=True)
merged_model = model.get_merged_model()
merged_model.save_pretrained(TARGET_PATH, max_shard_size="5GB")
model.tokenizer.save_pretrained(TARGET_PATH)
if model.processor is not None:
    model.processor.save_pretrained(TARGET_PATH)

print(f"\n🎉 HOÀN TẤT 100%! Model uncensored đã được lưu thành công trên Google Drive tại: {TARGET_PATH}")

In [ ]:
# @title 4. Kiểm tra Model đã được lưu trên Google Drive
import os

if os.path.exists(TARGET_PATH) and os.path.exists(os.path.join(TARGET_PATH, "config.json")):
    print("🎉 CHÚC MỪNG! Model đã được Uncensor và lưu thành công 100% trên Google Drive!")
    print(f"👉 Đường dẫn lưu trữ: {TARGET_PATH}")
    print("\nDanh sách các file đã tạo:")
    for f in sorted(os.listdir(TARGET_PATH)):
        size_mb = os.path.getsize(os.path.join(TARGET_PATH, f)) / (1024 * 1024)
        print(f"  - {f} ({size_mb:.1f} MB)")
else:
    print(f"⚠️ Đang kiểm tra thư mục: {TARGET_PATH}")
    if os.path.exists(TARGET_PATH):
        print("📁 Files hiện có:", os.listdir(TARGET_PATH))
    else:
        print("❌ Chưa tìm thấy thư mục model. Vui lòng kiểm tra lại log chạy ở Cell 3.")

In [ ]:
# @title 5. (Tùy chọn) Đẩy Model lên Hugging Face Hub
# @markdown Dán Access Token Hugging Face của bạn (bắt đầu bằng hf_...):
HF_TOKEN = "" # @param {type:"string"}

if HF_TOKEN.strip() and os.path.exists(TARGET_PATH):
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN.strip())
    try:
        username = api.whoami()["name"]
        print(f"👤 Tài khoản Hugging Face: {username}")
    except Exception:
        username = "Leon234aamon"
        print(f"👤 Sử dụng tài khoản: {username}")

    HF_REPO_ID = f"{username}/{OUTPUT_MODEL_NAME}"

    print(f"📤 Đang tạo repo private và tải model lên: {HF_REPO_ID}...")
    api.create_repo(repo_id=HF_REPO_ID, exist_ok=True, private=True)
    api.upload_folder(
        folder_path=TARGET_PATH,
        repo_id=HF_REPO_ID,
        repo_type="model"
    )
    print(f"🎉 Upload hoàn tất thành công! Xem model tại: https://huggingface.co/{HF_REPO_ID}")
else:
    print("ℹ️ Bỏ qua upload Hugging Face (Model đã được lưu an toàn trên Google Drive của bạn).")